:# Divye FE — Smoothed Target Encoding

Same as `s6e2-divye-fe.ipynb` but with **additive (Bayesian) smoothing** on the target encoding.

Raw TE formula: `group_mean`  
Smoothed TE formula: `(n × group_mean + k × global_mean) / (n + k)`

Where `n` = count of rows in the group, `k` = smoothing strength (higher = more shrinkage toward global mean).

- Small groups (rare values) → shrink toward global mean (reduces noise)
- Large groups (common values) → mostly unchanged

On 630k rows most groups are large, so the effect may be small — but worth testing.  
Testing `k=10` (mild) and `k=50` (stronger).  
Baseline: catboost_divye_fe OOF = 0.95566

In [1]:
import subprocess
import numpy as np
import pandas as pd
from itertools import combinations
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

DATA_DIR = Path('data')
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
ss    = pd.read_csv(DATA_DIR / 'sample_submission.csv')

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    if 'heart_disease' in df.columns:
        df['heart_disease'] = df['heart_disease'].map({'Absence': 0, 'Presence': 1})
    return df

train = prep(train)
test  = prep(test)

CAT_FEATURES = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
                'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']
NUM_FEATURES = ['age', 'bp', 'cholesterol', 'max_hr', 'st_depression']
ALL_FEATURES = CAT_FEATURES + NUM_FEATURES

X      = train[ALL_FEATURES]
y      = train['heart_disease'].values
X_test = test[ALL_FEATURES]

SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f'Train: {X.shape}  Test: {X_test.shape}')

Train: (630000, 13)  Test: (270000, 13)


In [2]:
def compute_freq_features(train_df, test_df, cols):
    combined = pd.concat([train_df[cols], test_df[cols]])
    tr_freq, te_freq = {}, {}
    for col in cols:
        freq_map = combined[col].value_counts(normalize=True)
        tr_freq[f'{col}_freq'] = train_df[col].map(freq_map).fillna(0).values
        te_freq[f'{col}_freq'] = test_df[col].map(freq_map).fillna(0).values
    return tr_freq, te_freq


def compute_oof_te_smoothed(train_df, test_df, cols, y, skf, k=10):
    """OOF target encoding with additive smoothing.
    Smoothed TE = (n * group_mean + k * global_mean) / (n + k)
    k=10: mild smoothing; k=50: stronger.
    """
    global_mean = y.mean()
    oof_te  = {f'{col}_te': np.zeros(len(train_df)) for col in cols}
    test_te = {f'{col}_te': np.zeros(len(test_df))  for col in cols}

    for col in cols:
        key     = f'{col}_te'
        tr_vals = train_df[col].values
        te_vals = test_df[col].values
        fold_test = []

        for tr_idx, val_idx in skf.split(train_df, y):
            stats = (pd.DataFrame({'v': tr_vals[tr_idx], 't': y[tr_idx]})
                       .groupby('v')['t']
                       .agg(['mean', 'count']))
            smoothed = (stats['count'] * stats['mean'] + k * global_mean) / (stats['count'] + k)
            oof_te[key][val_idx] = (pd.Series(tr_vals[val_idx])
                                      .map(smoothed).fillna(global_mean).values)
            fold_test.append(pd.Series(te_vals).map(smoothed).fillna(global_mean).values)

        test_te[key] = np.mean(fold_test, axis=0)

    return oof_te, test_te


def select_top_interactions(oof_te, y, n=8):
    corrs = {col: abs(np.corrcoef(vals, y)[0, 1]) for col, vals in oof_te.items()}
    top = sorted(corrs, key=corrs.get, reverse=True)[:n]
    print('Top TE features by |corr|:')
    for col in top:
        print(f'  {col:40s}  {corrs[col]:.4f}')
    return top


def make_interaction_features(te_dict, top_cols):
    return {f'{c1}_x_{c2}': te_dict[c1] * te_dict[c2]
            for c1, c2 in combinations(top_cols, 2)}


def build_augmented(base_df, freq_dict, te_dict, inter_dict):
    df = base_df.copy().reset_index(drop=True)
    for name, vals in {**freq_dict, **te_dict, **inter_dict}.items():
        df[name] = vals
    return df


SMOOTH_K = 10

print(f'Smoothing k={SMOOTH_K}')
print('Computing frequency encoding...')
tr_freq, te_freq = compute_freq_features(X, X_test, ALL_FEATURES)

print('Computing smoothed OOF target encoding...')
oof_te, test_te = compute_oof_te_smoothed(X, X_test, ALL_FEATURES, y, SKF, k=SMOOTH_K)

print('\nSelecting top-8 for interactions...')
top8 = select_top_interactions(oof_te, y, n=8)

tr_inter = make_interaction_features(oof_te,  top8)
te_inter = make_interaction_features(test_te, top8)
print(f'{len(tr_inter)} interaction features')

Smoothing k=10
Computing frequency encoding...
Computing smoothed OOF target encoding...

Selecting top-8 for interactions...
Top TE features by |corr|:
  thallium_te                               0.6058
  chest_pain_type_te                        0.5252
  max_hr_te                                 0.4784
  number_of_vessels_fluro_te                0.4632
  st_depression_te                          0.4488
  exercise_angina_te                        0.4419
  slope_of_st_te                            0.4298
  sex_te                                    0.3424
28 interaction features


In [3]:
CATBOOST_PARAMS = dict(
    iterations=2000, learning_rate=0.05, depth=6,
    task_type='CPU', thread_count=-1,
    cat_features=CAT_FEATURES, random_seed=42, verbose=0,
)

global_mean = y.mean()
oof_preds   = np.zeros(len(y))
test_folds  = np.zeros((5, len(X_test)))

for fold, (tr_idx, val_idx) in enumerate(SKF.split(X, y)):
    print(f'\n=== Fold {fold + 1}/5 ===')
    X_tr_base  = X.iloc[tr_idx].reset_index(drop=True)
    X_val_base = X.iloc[val_idx].reset_index(drop=True)
    y_tr, y_val = y[tr_idx], y[val_idx]

    fold_tr_freq, fold_te_freq  = compute_freq_features(X_tr_base, X_test,    ALL_FEATURES)
    _,            fold_val_freq = compute_freq_features(X_tr_base, X_val_base, ALL_FEATURES)

    fold_tr_te, fold_val_te, fold_te_te = {}, {}, {}
    for col in ALL_FEATURES:
        key   = f'{col}_te'
        stats = (pd.DataFrame({'v': X_tr_base[col].values, 't': y_tr})
                   .groupby('v')['t'].agg(['mean', 'count']))
        smoothed = (stats['count'] * stats['mean'] + SMOOTH_K * global_mean) / (stats['count'] + SMOOTH_K)
        fold_tr_te[key]  = X_tr_base[col].map(smoothed).fillna(global_mean).values
        fold_val_te[key] = X_val_base[col].map(smoothed).fillna(global_mean).values
        fold_te_te[key]  = X_test[col].map(smoothed).fillna(global_mean).values

    fold_tr_inter  = make_interaction_features(fold_tr_te,  top8)
    fold_val_inter = make_interaction_features(fold_val_te, top8)
    fold_te_inter  = make_interaction_features(fold_te_te,  top8)

    X_tr_aug  = build_augmented(X_tr_base,  fold_tr_freq,  fold_tr_te,  fold_tr_inter)
    X_val_aug = build_augmented(X_val_base, fold_val_freq, fold_val_te, fold_val_inter)
    X_te_aug  = build_augmented(X_test,     fold_te_freq,  fold_te_te,  fold_te_inter)

    m = CatBoostClassifier(**CATBOOST_PARAMS)
    m.fit(X_tr_aug, y_tr, eval_set=(X_val_aug, y_val), early_stopping_rounds=50)

    oof_preds[val_idx] = m.predict_proba(X_val_aug)[:, 1]
    test_folds[fold]   = m.predict_proba(X_te_aug)[:, 1]
    print(f'Fold {fold+1}  AUC={roc_auc_score(y_val, oof_preds[val_idx]):.5f}  best_iter={m.best_iteration_}')

cv_auc     = roc_auc_score(y, oof_preds)
test_preds = test_folds.mean(axis=0)

print(f'\nOOF AUC (smoothed k={SMOOTH_K}): {cv_auc:.5f}')
print(f'Baseline divye_fe:               0.95566')
print(f'Delta:                           {cv_auc - 0.95566:+.5f}')


=== Fold 1/5 ===
Fold 1  AUC=0.95607  best_iter=493

=== Fold 2/5 ===
Fold 2  AUC=0.95490  best_iter=504

=== Fold 3/5 ===
Fold 3  AUC=0.95575  best_iter=688

=== Fold 4/5 ===
Fold 4  AUC=0.95535  best_iter=585

=== Fold 5/5 ===
Fold 5  AUC=0.95618  best_iter=619

OOF AUC (smoothed k=10): 0.95565
Baseline divye_fe:               0.95566
Delta:                           -0.00001


In [ ]:
np.save('submissions/oof_divye_fe_smoothed.npy', oof_preds)

label = f'catboost_divye_fe_smooth{SMOOTH_K}'
fname = f'submissions/{label}.csv'
sub = ss.copy()
sub['Heart Disease'] = test_preds
sub.to_csv(fname, index=False)

desc = f'{label} | cv_auc={cv_auc:.4f}'
print(f'Saved: {fname}')
print(f'desc:  {desc}')
print(f'OOF AUC: {cv_auc:.5f}  (baseline divye_fe: 0.95566  delta: {cv_auc - 0.95566:+.5f})')

In [ ]:
import subprocess
r = subprocess.run(
    ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
     '-f', fname, '-m', desc],
    capture_output=True, text=True
)
status = 'submitted' if 'successfully' in r.stdout.lower() else r.stdout.strip()[:120]
print(f'{label}: {status}')